# Class 5 (gentle version): Can We Just *Ask* an AI Instead?

### 🧭 The one question we are answering

In **Class 1** you built a classifier. In **Class 2** you built a regressor. Both times the
recipe was the same: *collect labelled data → train a model → check it on held-out data.*

Today we try something that skips the first two steps entirely. We will **just ask a
language model** — the same kind of model behind ChatGPT and Claude — to do those two jobs,
with **no training data at all**, and then ask the only question that matters:

> **Is it any good — and when should I still build the small, boring model instead?**

### 🗺️ Roadmap

| Part | What we do | Time |
|---|---|---|
| 1 | Say hello to a real model | 10 min |
| 2 | **Classification** without training — sentiment of customer reviews | 25 min |
| 3 | **Regression** without training — the same house prices as Class 2 | 25 min |
| 4 | When to use which, and what it costs | 10 min |

### 🐍 About the Python

This notebook is written for people who **have not written much Python**. Every task asks you
to change *a sentence in quotes* — never to write a loop. If a cell looks long, you can
still just press ▶︎ and read the output; nothing here needs to be memorised.

> **How to run a cell:** click it, then press **Shift + Enter**. Run them **in order**,
> top to bottom.


In [ ]:
# If running on Google Colab, clone the repo (if needed),
# move into the repo directory, and ensure it’s on the Python path.

import sys, os

def in_colab():
    try: import google.colab; return True
    except: return False

if in_colab():
    repo = "Hands-On-Notebooks"
    if os.path.basename(os.getcwd()) != repo:
        if not os.path.exists(repo):
            !git clone https://github.com/BridgingAISocietySummerSchools/{repo}
        %cd {repo}
    if '.' not in sys.path:
        sys.path.append('.')

In [ ]:
# Setup — run this once. It imports our tools and the example data.

import numpy as np
from sklearn.linear_model import LinearRegression

from plotting_utils.llm_simple import (
    REVIEWS,                    # 8 customer reviews with agreed-on labels
    HOUSE_SIZES, HOUSE_PRICES,  # the same 10 houses as Class 2
    to_label,                   # "clearly positive!" -> "positive"
    to_number,                  # "about 7 out of 10"  -> 7.0
    to_price_in_thousands,      # "about $350k"        -> 350.0
    show_label_results,
    plot_accuracy_bars,
    plot_llm_vs_line,
    show_error_comparison,
    plot_repeat_spread,
    create_classifier_playground,
)

print("✅ Ready.")

### 🔑 Connect to a Real Model

Everything today talks to a **real language model** through
[OpenRouter](https://openrouter.ai/). **Your instructor will give you a key for the session.**

> ⚠️ **Never type an API key into a notebook cell.** Notebooks get saved, committed,
> screenshotted and shared — and a key in a notebook is a key on the internet.

The cell below asks for the key in a **hidden input box**, so it stays in memory and never
lands in the saved file. If a key is already in a `.env` file or in Colab's 🔑 **Secrets**
panel, it is picked up automatically and you will not be prompted.

The loading logic lives in [`llm_client.py`](llm_client.py) — short, and worth a look.

In [ ]:
import os, getpass
from llm_client import ask_llm, llm_available, describe_setup, print_usage

if not llm_available():
    try:
        key = getpass.getpass("Paste the workshop key (hidden): ").strip()
    except Exception:                 # no interactive input available
        key = ""
    if key:
        os.environ["OPENROUTER_API_KEY"] = key

LLM_READY = describe_setup()

## Part 1: Say Hello ⏱️ 10 min

One function does everything in this notebook: **`ask`**. You give it a question as text,
it gives you back the model's answer as text. That is the entire interface.

In [ ]:
def ask(question):
    """Send a question to the model and return its answer as text."""
    return ask_llm(question, max_tokens=200)


print(ask("In two sentences, what is machine learning?"))

### ✏️ Task 1 — Ask it something yourself (3 min)

Change the sentence inside the quotes below, then run the cell. Try a question about **your
own field**. Then try one where you already know the right answer — and check it.

In [ ]:
# ✏️ TASK 1 — change the text inside the quotes, then run the cell.

print(ask("Explain what a p-value is to someone who has never studied statistics."))

### 💬 One Thing to Notice Now

You wrote **no rules** and gave **no examples**. There was no training step, no data set,
no accuracy score. That is genuinely new — and it is also exactly why we have to be careful.
A fluent answer is not the same as a correct one, and nothing in that output told you which
one you got.

Keep that in mind for the next two parts, where we can finally *measure*.

## Part 2: Classification Without Training ⏱️ 25 min

### 🔁 What Class 1 needed

To build the disease classifier in Class 1, we needed:

- **10,000 labelled patients** — someone had to establish the true diagnosis for every one
- a **train/test split**, so we could judge the model honestly
- a **training step** that fitted the model's parameters to that data
- and a model that only ever answers **that one question**

### 🎯 What we are going to do instead

Ask. In English. Once.

In [ ]:
review = "The battery lasts all day and the screen is gorgeous. Best purchase this year."

reply = ask(f"Is this customer review positive or negative? Answer with one word.\n\n{review}")
print(reply)

### 🤔 Fine — But Is It Actually Any Good?

That worked. It would also have "worked" if the model had been wrong, and we would not have
known. So we do the one thing Class 1 drilled into us: **we check it against labelled data.**

Here are eight reviews, each with a label we agreed on in advance. Notice that the model
never sees these labels — they exist only so we can grade it, exactly like the test set in
Class 1.

> Note that we still need labelled data. Not to *train* the model — to **trust** it.

In [ ]:
for r in REVIEWS:
    print(f"{r['label']:>8}  |  {r['text']}")

### 🏷️ A Reusable Classifier, in Six Lines

Below is the whole thing. Two details are worth your attention:

- **`Answer with the category name only.`** Without it the model writes a paragraph, and a
  paragraph is not a label. Telling it the *shape* of the answer you want is most of prompting.
- **`to_label(...)`** takes the reply and works out which category it named, so
  `"Clearly positive."` and `"positive"` both count. A trained model hands you a number;
  a language model hands you prose, and turning prose into data is work you will always have to do.

In [ ]:
def classify(text, labels):
    """Ask the model to sort `text` into exactly one of `labels`."""
    prompt = (f"Classify the text into exactly one of these categories: {', '.join(labels)}.\n"
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    reply = ask_llm(prompt, max_tokens=20)
    return to_label(reply, labels)


print(classify("This thing is fantastic!", ["positive", "negative"]))

### 📊 Now Grade It on All Eight

The next cell makes **8 real API calls**, one per review, so give it ~20 seconds.

In [ ]:
llm_accuracy = None                       # so the next cells work even without a key

if not LLM_READY:
    print("⏭️  This cell needs an API key — see the setup cell near the top.")
else:
    guesses = []
    for r in REVIEWS:
        guesses.append(classify(r["text"], ["positive", "negative"]))
        print("·", end="")                # a dot per call, so you can see progress
    print()

    llm_accuracy = show_label_results(
        [r["text"] for r in REVIEWS],
        [r["label"] for r in REVIEWS],
        guesses,
    )

### 💬 Read That Table Carefully

Two things are worth arguing about:

**1. Look at *which* ones it missed.** The first five reviews are easy — they say "gorgeous"
and "best purchase", or "broken" and "stopped working". The last three are the interesting
ones:

| Review | Why it's hard |
|---|---|
| *"if you enjoy reading a 60-page manual…"* | **Sarcasm.** Every individual word is positive. |
| *"sound quality is excellent, but the app crashes"* | **Good then bad.** Which half is the verdict? |
| *"shipping was slow, but worth the wait"* | **Bad then good.** Same problem, opposite order. |

**2. Our labels are not facts.** We *decided* that "excellent sound, daily crashes" is
negative. A room full of people would not fully agree. When a model disagrees with your
label, the honest first question is not "why is the model wrong?" but **"was my label ever
right?"** — a question worth asking of every data set you will ever build.

### ⚖️ And now the comparison that matters

| | Class 1's classifier | Today's model |
|---|---|---|
| Labelled examples needed to **build** it | 10,000 | **0** |
| Labelled examples needed to **trust** it | ~2,000 (the test set) | 8 (and more would be better) |
| Time to build | a training run | one sentence |
| Can also sort emails by topic? | ❌ needs a new data set and retraining | ✅ change the prompt |

That last row is the one to remember, and it is the whole point of Task 3.

### 🪤 Wait — Compare It To Something Dumb First

Class 1 taught the most important habit in this whole course: **a percentage means nothing on
its own.** There, a model that predicted "healthy" for every single patient scored 90%
accuracy and helped nobody.

So before we are impressed, let's beat the model up with two embarrassing baselines on the
**same eight reviews**:

- **"Always say positive"** — no reading required.
- **Keyword counting** — count nice words, count nasty words, take the bigger pile. Ten lines,
  no AI, runs in a microsecond, costs nothing.


In [ ]:
# The dumbest possible baselines, for scale. Always check these first.

GOOD = ["great", "gorgeous", "excellent", "flawlessly", "best", "worth", "nice", "wonderful"]
BAD = ["broken", "cheap", "stopped", "crashes", "slow", "never"]

def keyword_rule(text):
    """No AI at all: count nice words vs nasty words."""
    t = text.lower()
    return "positive" if sum(w in t for w in GOOD) > sum(w in t for w in BAD) else "negative"


truth = [r["label"] for r in REVIEWS]
scores = {
    "Always say 'positive'": sum(t == "positive" for t in truth) / len(truth),
    "Keyword counting": sum(keyword_rule(r["text"]) == t for r, t in zip(REVIEWS, truth)) / len(truth),
}
if llm_accuracy is not None:
    scores["Language model"] = llm_accuracy

plot_accuracy_bars(scores, title="Sentiment on the same 8 reviews")

### 💬 That Is the Number That Matters

The language model should be ahead — but notice **how far ahead**, and what it is ahead *of*.
Keyword counting is free, instant, and completely transparent, and it is not embarrassingly
worse. On eight reviews, a one-review difference is 12.5 percentage points, which is well
inside the noise.

> 📌 **The honest question is never "is the AI good?" but "is the AI enough better than the
> cheap thing to be worth the cost?"** You cannot answer that without measuring the cheap
> thing, and almost nobody does.

That is Class 1's lesson, unchanged, applied to a completely different kind of model.

### ✏️ Task 2 — Try to break it (5 min)

Replace the reviews below with your own. **Try to fool it.** Sarcasm and faint praise
("it's certainly a product") are your best weapons. Write a review *you* find hard to label,
and see whether it struggles in the same place you did.

In [ ]:
# ✏️ TASK 2 — edit these sentences (add or remove lines), then run the cell.

my_reviews = [
    "Well, it arrived. Eventually.",
    "I have owned worse toasters.",
    "The colour is nice.",
]

for text in my_reviews:
    print(f"{classify(text, ['positive', 'negative']):>8}  |  {text}")

### ✏️ Task 3 — The thing a trained model cannot do ⭐ (7 min)

Class 1's classifier answers exactly one question: *does this patient have the disease?*
To make it sort support emails instead, you would need a new labelled data set and a new
training run — **days of work**.

Here, you change a sentence. Put in **any** text and **any** categories and press *Run*.
This is the single biggest practical difference between the two approaches.

In [ ]:
create_classifier_playground(classify)

## Part 3: Regression Without Training ⏱️ 25 min

Classification asked *which category?* Regression asks **how much?** — a number.

These are the **same ten houses** from Class 2, so this is a fair fight. First, the
Class 2 answer: fit a straight line through the data.

In [ ]:
sizes = np.array(HOUSE_SIZES).reshape(-1, 1)
prices = np.array(HOUSE_PRICES)

line = LinearRegression().fit(sizes, prices)
line_guesses = line.predict(sizes)

print(f"Class 2's rule: price = {line.coef_[0]:.3f} × size + {line.intercept_:.1f}  (thousands of $)")
print(f"That is ${line.coef_[0] * 1000:.0f} per extra square foot.")

### 🏠 Now Ask the Model Instead

Same job, no training. Two details again worth noticing in the prompt:

- We tell it the **unit** (*thousands of dollars*) and the **format** (*just a number*).
  Ambiguity here is how you end up with `"$350,000"` in a column meant to hold `350`.
- We give it **no house prices at all**. It has never seen this data set. It is working
  purely from what it picked up about houses while reading the internet.

In [ ]:
def predict_price(size):
    """Ask the model what a house of this many square feet costs."""
    prompt = (f"A house is {size} square feet. Estimate its price in thousands of US dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    reply = ask_llm(prompt, max_tokens=20)
    return to_price_in_thousands(reply)


print(predict_price(1800), "thousand dollars for an 1,800 sq ft house")

### 📊 Ten Houses, Two Methods

Another **10 API calls** — about 20 seconds.

In [ ]:
if not LLM_READY:
    print("⏭️  This cell needs an API key — see the setup cell near the top.")
else:
    llm_guesses = []
    for size in HOUSE_SIZES:
        llm_guesses.append(predict_price(size))
        print("·", end="")
    print()

    plot_llm_vs_line(HOUSE_SIZES, HOUSE_PRICES, llm_guesses, line_guesses)
    errors = show_error_comparison(HOUSE_PRICES, llm_guesses, line_guesses)

### 💬 Who Won — and Why the Line Is Still the Right Default

Usually the boring little line beats the frontier model here, and often it is not close.
But look at *your* numbers above rather than trusting mine: the model's answer depends
entirely on which housing market it happened to assume, so it can land close by luck.
Either way, three things are true:

1. **The line saw the answers; the model did not.** The line was fitted to *these* ten
   houses, so its error is measured on its own homework — a flattering test. The model has
   never seen them and has no idea which city, year, or market we mean. If the model came
   close, that is a coincidence about the market it guessed, not knowledge of ours.
2. **Numbers are not what language models are built for.** It is predicting the *text* of a
   plausible price, not calculating one.
3. **The line gives you a rule you can read.** `$167 per square foot` is a sentence you can
   take to a colleague, argue with, and check. Ask the model *why* it said 350 and you get a
   fluent paragraph that may have nothing to do with how it actually produced the number.

> 📌 **If you have a table of numbers and a column you want to predict, train the small model.**
> This is still true, it is still the right default, and it is not going to stop being true.

### 🎲 One More Problem: It Doesn't Always Say the Same Thing

Ask the fitted line about an 1,800 sq ft house a thousand times and you get the identical
number a thousand times. Language models have a **"creativity" dial** (called *temperature*).
Everything so far ran with it turned to zero. Let's turn it up and ask the *same question*
five times.

In [ ]:
if not LLM_READY:
    print("⏭️  This cell needs an API key — see the setup cell near the top.")
else:
    prompt = ("A house is 1800 square feet. Estimate its price in thousands of US dollars.\n"
              "Answer with just a number. Example: 250")

    repeats = [to_price_in_thousands(ask_llm(prompt, max_tokens=20, temperature=1.0))
               for _ in range(5)]

    print("The five answers:", repeats)
    plot_repeat_spread(repeats)
    print(f"\nFor comparison, Class 2's line says {line.predict(np.array([[1800]]))[0]:.0f} — every single time.")

### 💬 Why This Is a Real Engineering Problem

If the same input can give different outputs, then **"it worked when I tried it" stops being
evidence.** Testing, debugging, auditing, and reproducing a result all get harder. Two
practical consequences:

- For anything where you want a stable answer, **set temperature to 0** (our `ask` does).
  It reduces the wobble a great deal, but it is not a mathematical guarantee.
- If a decision matters, **ask more than once** and look at the spread — the way you would
  with any noisy measurement.

> 🔎 See a `None` in that list of five answers? That is a reply that contained no number at
> all — the model wrote a sentence where we asked for a digit. It gets skipped rather than
> guessed at. Expect this: **a few percent of replies will not match the format you asked
> for**, and real systems need a plan for those, not an exception.

### ✏️ Task 4 — The case where the model wins ⭐ (7 min)

So far the line won. But notice what the line *required*: a table of past examples with the
right answer filled in. Now consider a task where **no such table exists**:

> *"How urgent is this email, on a scale of 1 to 10?"*

Nobody has a labelled data set of 10,000 emails rated for urgency. Building one would take
weeks. Yet the model will give you a sensible number immediately — and *that* is the trade
this whole notebook is about.

In [ ]:
# ✏️ TASK 4 — edit these messages, then run the cell.

def rate_urgency(message):
    prompt = (f"How urgent is this message, on a scale of 1 (not at all) to 10 (drop everything)?\n"
              f"Answer with just a number.\n\n{message}")
    return to_number(ask_llm(prompt, max_tokens=20))


messages = [
    "Reminder: the office plant rota starts Monday.",
    "The production database is down and customers cannot log in.",
    "Following up on my invoice from last month.",
]

for m in messages:
    print(f"{rate_urgency(m):>5}  |  {m}")

### 💬 What Did You Just Build?

A working urgency scorer, in four lines, with **zero labelled examples**. Now the hard
question: **would you route your company's support queue with it?**

Some things you would want first, and none of them are exotic:

- A few hundred messages rated by **humans**, to measure agreement — the Part 2 move again.
- A check that it is not **systematically harsh** on messages written in a second language,
  or in ALL CAPS, or by a particular customer segment.
- A rule for what happens when it is **wrong**, because it will be.

Which is the same list Class 1 gave you for the disease classifier. **The tooling changed
completely. The obligations did not.**

## Part 4: So Which One Should I Use? ⏱️ 10 min

| | **Train a small model** (Classes 1–2) | **Ask a language model** (today) |
|---|---|---|
| **Labelled data needed** | thousands of rows | none to start |
| **Time to a first version** | days | minutes |
| **Cost per prediction** | ~free | a fraction of a cent, but **not** free |
| **Speed per prediction** | microseconds | ~1–3 seconds |
| **Same answer every time?** | ✅ always | ⚠️ usually, not guaranteed |
| **Can you read the rule?** | ✅ `$167 per sq ft` | ❌ a fluent story, not the reason |
| **Best on tables of numbers** | ✅ | ❌ |
| **Best on free text, images, messy input** | ❌ | ✅ |
| **Change the task** | retrain from scratch | rewrite one sentence |

### 🧭 A Rule of Thumb You Can Actually Use

- **Numbers in a spreadsheet, and you have history?** → train the small model. Cheaper,
  faster, steadier, explainable.
- **Text, images, or a task nobody has labelled yet?** → start with the language model.
- **Millions of predictions a day?** → the small model, on cost and latency alone.
- **Not sure the task is even worth doing?** → prompt it this afternoon to find out, *then*
  decide whether to build the real thing. This is the most underrated use of these models:
  a **one-hour feasibility test** instead of a three-week project.

### 💰 What Today Actually Cost

Nothing above is free. Let's look at the bill.

In [ ]:
print_usage()

### 💬 Put That Number in Context

Scale up whatever you just saw. If a support inbox gets **10,000 messages a day** and each
one costs a fraction of a cent to classify, that is real money every month — for a job the
Class 1 classifier would do for essentially nothing, *once someone labels the data*.

That trade — **your labelling time up front, or a per-prediction fee forever** — is the
actual decision behind most "should we use AI for this?" conversations.

## 🎓 Wrap-Up: Four Ideas Worth Keeping

1. **You can do classification and regression with no training data at all.** Write the task
   in English, state the format you want back, and read the answer. For text especially,
   this is often good enough to be useful on day one.

2. **You still need labelled data — to *trust* it, not to *train* it.** Every honest claim
   in this notebook came from those eight hand-labelled reviews and ten known prices. Skip
   that step and you have a demo, not a system.

3. **For tables of numbers, the small boring model is still better.** Cheaper, faster,
   identical every time, and it hands you a rule you can read and argue with. Reach for the
   language model when the input is messy or the labels don't exist yet.

4. **The obligations survive the change of tooling.** Measure it, check who it fails, plan
   for being wrong. Class 1's warnings about accuracy, bias and honest evaluation apply
   *exactly* as much to a prompt as to a trained classifier.

### 🧭 Discussion Questions

1. We labelled *"excellent sound, but crashes daily"* as negative. Defend the opposite label.
   What does the disagreement tell you about reported accuracy numbers in general?
2. The model has never seen our houses, yet produced plausible prices. Where did those
   numbers come from — and what happens if our houses are in a market unlike the ones it read about?
3. Your team must classify 50,000 support tickets a month. Argue both sides, then choose.
4. Task 4 built an urgency scorer with no labelled data. What is the **worst** thing that
   could happen if you shipped it on Monday, and what would you check first?

### ✅ Before You Use This On Something Real

- [ ] Do I have labelled examples to **measure** it, not just a few I eyeballed?
- [ ] Do I know **which cases** it fails on — and are they the cases that matter most?
- [ ] Have I checked it is not systematically worse for a particular **group** of inputs?
- [ ] Do I know the **cost per prediction** at my real volume?
- [ ] Have I compared against the **boring baseline** — a small trained model, or even a
      keyword rule? Sometimes it wins, and it is embarrassing to find that out late.
- [ ] Is there a plan for when it is **wrong**?

### 📚 Where to Go Next

- [`05_agentic_ai.ipynb`](05_agentic_ai.ipynb) — the full Class 5: **RAG**, giving a model
  your *own* documents so it stops guessing, plus tool-using **agents** and prompt injection.
- [`self_learning/05_agentic_ai.ipynb`](self_learning/05_agentic_ai.ipynb) — a ~3-hour
  deep dive that builds a small language model from scratch.